# 06: Multistream Pipeline

This notebook extends the work done in Notebook 01 and Notebook 02. It focuses on building a multistream data pipeline that can handle multiple synchronized streams from the VRS recording.

This next notebook will generalize that approach from one stream to all relevant streams.

## In this notebook, we will:

1. Inventory all streams (Reuse the discovery logic from Notebook 01)
- Build a canonical table with stream id, label, modality, sample count, and time coverage.
- Separate image, IMU, and auxiliary streams clearly.

2. Validate per-stream access
- Confirm that each stream can be decoded or read by index.
- Measure timestamp monotonicity and sample spacing per stream.
- Flag empty streams, sparse streams, and streams with non-standard timing.

3. Define a multistream dataset abstraction
- Extend the single-stream dataset pattern from Notebook 02.
- Decide whether one item should return a dictionary of synchronized streams or a per-stream record bundle.
- Keep the API compatible with PyTorch `Dataset` and `DataLoader`.

4. Build synchronization and sampling rules
- Align streams on a shared time axis or a reference stream.
- Define how to handle missing samples, different rates, and partial overlap.
- Add deterministic indexing rules for training and analysis use cases.

5. Add visualization and sanity checks
- Display frame-aligned samples across multiple streams.
- Inspect synchronization quality on a few representative windows.
- Verify that the outputs are physically consistent before exporting code.

6. Export final artifacts
- Save stream tables, quality reports, and any derived indices under `data/processed/`.
- Document the chosen multistream data model and any assumptions about synchronization.

7. Package reusable code into `scripts/`
- Move stable helpers into Python modules following package conventions.
- Separate stream discovery, indexing, dataset definitions, and transforms into dedicated files.
- Keep notebooks focused on experimentation and validation, not on long-term implementation.

As a sample, we will use `kettle_and_forklift_recording.vrs` and its associated `kettle_and_forklift_recording.json` metadata, located in `data/raw/kettle_and_forklift`. The goal is to create a robust multistream pipeline that can be easily adapted to other recordings.

## 6.1 Setup and Recording Discovery

This section prepares the notebook for multistream work.
We reuse the same style established in Notebook 01 and Notebook 02:
- validate the recording paths
- make the project modules importable
- open the VRS provider
- discover the available streams as the first canonical inventory step

The goal is to keep this notebook focused on the multistream extension of the pipeline, while preserving a clean separation between exploration and reusable code.

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from projectaria_tools.core import data_provider, sensor_data

# Define the sample recording used for the multistream pipeline.
DATA_DIR = Path("..") / "data" / "raw" / "kettle_and_forklift"
VRS_PATH = DATA_DIR / "kettle_and_forklift_recording.vrs"
JSON_PATH = DATA_DIR / "kettle_and_forklift_recording.json"
OUT_DIR = Path("..") / "data" / "processed" / "kettle_and_forklift"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Validate the required input files before continuing.
if not VRS_PATH.exists():
    raise FileNotFoundError(f"VRS file not found: {VRS_PATH.resolve()}")
if not JSON_PATH.exists():
    raise FileNotFoundError(f"JSON metadata file not found: {JSON_PATH.resolve()}")

print("VRS path:", VRS_PATH.resolve())
print("JSON path:", JSON_PATH.resolve())
print("Output dir:", OUT_DIR.resolve())

# Make the repository root importable so helper modules in scripts/ can be reused.
project_root = next((p for p in [Path.cwd().resolve(), Path.cwd().resolve().parent] if (p / "scripts" / "aria_dataset.py").exists()), None)
if project_root is None:
    raise RuntimeError("Could not find the project root containing scripts/aria_dataset.py")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Reuse the frame extraction helper established in Notebook 02.
from scripts.aria_dataset import _extract_image_array_from_sample

# Create the VRS provider used by the rest of the notebook.
provider = data_provider.create_vrs_data_provider(str(VRS_PATH))
if provider is None:
    raise RuntimeError("Failed to initialize the VRS data provider.")

print("Provider initialized successfully.")
print("Available streams:", len(provider.get_all_streams()))

## 6.2 Stream Inventory and Canonical Table

This section recreates the stream discovery pattern from Notebook 01, but with a multistream perspective.
We want one canonical table that makes the recording structure explicit before any dataset or synchronization logic is introduced.

The table will be the base artifact for the rest of the notebook: every later decision about sampling, alignment, and packaging should refer back to this inventory.

In [ ]:
# Helper to classify streams into a broad modality category.
def infer_modality(label: str) -> str:
    """Infer a broad modality label from the stream name or label."""
    text = (label or "").lower()
    if any(key in text for key in ["camera", "image", "rgb", "slam", "et"]):
        return "image"
    if any(key in text for key in ["imu", "accel", "gyro"]):
        return "imu"
    if any(key in text for key in ["audio", "mic"]):
        return "audio"
    if any(key in text for key in ["gps", "baro", "mag", "bluetooth", "ble", "wifi", "wps"]):
        return "auxiliary"
    return "unknown"


# Build a canonical inventory table for all available streams.
stream_rows = []
for stream_id in provider.get_all_streams():
    stream_label = provider.get_label_from_stream_id(stream_id)
    sample_count = provider.get_num_data(stream_id)
    stream_rows.append({
        "stream_id": str(stream_id),
        "label": str(stream_label),
        "modality": infer_modality(str(stream_label)),
        "sample_count": int(sample_count),
    })

streams_df = pd.DataFrame(stream_rows).sort_values(["modality", "label", "stream_id"]).reset_index(drop=True)

print(f"Discovered {len(streams_df)} streams.")
display(streams_df)

# Keep a short summary that can be reused by later sections.
stream_summary_df = streams_df.groupby("modality", dropna=False).size().reset_index(name="num_streams")
display(stream_summary_df)